# 02 - Pull HTTP data into Spark and Iceberg Bronze

Start notebook 01 first and keep its API kernel running. Run this notebook in a separate Spark-enabled kernel. It includes all its own configuration and HTTP client code.

The installed Spark 3.5 runtime has no general REST SQL data source configured. Do not use `spark.read.format("http")` or treat an API response envelope as a JSON file path. This notebook demonstrates Python HTTP requests on the driver, followed by DataFrame creation, and HTTP requests inside bounded Spark partitions, followed by Spark JSON parsing.

In [ ]:
import os
import re
import socket
import json

LAB = "http_rest_lake"
assert re.fullmatch(r"[a-z][a-z0-9_]*", LAB)
API_BASE = os.getenv("REST_API_BASE_URL", "http://spark-master:18717").rstrip("/")
LAKE_ROOT = f"s3://warehouse/iceberg/{LAB}"
print("Namespace:", "polaris." + LAB)

## Start Spark with Polaris and SeaweedFS

In [ ]:
import os

import socket

REQUIRED_ENV = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "POLARIS_CLIENT_ID", "POLARIS_CLIENT_SECRET"]

def preflight():
    missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing existing credential environment names: " + ", ".join(missing))
    import importlib.util
    from pathlib import Path
    if importlib.util.find_spec("pyspark") is None:
        raise RuntimeError("Choose the existing Spark-enabled Python kernel; do not pip install PySpark")
    jars = Path(os.environ.get("SPARK_HOME", "/opt/spark")) / "jars"
    patterns = ["iceberg-spark-runtime-*.jar", "iceberg-aws-bundle-*.jar",
                "hadoop-aws-*.jar", "aws-java-sdk-bundle-*.jar"]
    absent = [pattern for pattern in patterns if not list(jars.glob(pattern))]
    if absent:
        raise RuntimeError("Missing image JARs; stop without rebuilding/installing: " + ", ".join(absent))
    print("PREFLIGHT_PASS: required environment names, PySpark and image JARs present")

def start_spark(app_name):
    preflight()
    from pyspark.sql import SparkSession
    if SparkSession.getActiveSession() is not None:
        raise RuntimeError("Stop the existing Spark session and restart this kernel before setup")
    builder = (SparkSession.builder.appName(app_name).master("spark://spark-master:7077")
        .config("spark.driver.host", os.environ.get("POD_IP") or socket.gethostbyname(socket.gethostname()))
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.cores.max", "1").config("spark.executor.cores", "1")
        .config("spark.driver.memory", "512m").config("spark.executor.memory", "512m").config("spark.sql.shuffle.partitions", "2")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.polaris.type", "rest")
        .config("spark.sql.catalog.polaris.uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog"))
        .config("spark.sql.catalog.polaris.warehouse", "training")
        .config("spark.sql.catalog.polaris.rest.auth.type", "oauth2")
        .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")
        .config("spark.sql.catalog.polaris.oauth2-server-uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog").rstrip("/") + "/v1/oauth/tokens")
        .config("spark.sql.catalog.polaris.header.Polaris-Realm", "POLARIS")
        .config("spark.sql.catalog.polaris.rest-metrics-reporting-enabled", "false")
        .config("spark.sql.catalog.polaris.credential", os.environ["POLARIS_CLIENT_ID"] + ":" + os.environ["POLARIS_CLIENT_SECRET"])
        .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.polaris.s3.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.sql.catalog.polaris.s3.path-style-access", "true")
        .config("spark.sql.catalog.polaris.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.sql.catalog.polaris.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.sql.catalog.polaris.client.region", "us-east-1")
        .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential"))
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print("SPARK_STARTED", spark.version, spark.sparkContext.applicationId)
    return spark


from pyspark.sql import functions as F
spark = start_spark("rest-" + LAB)
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS polaris.{LAB} LOCATION '{LAKE_ROOT}'")

## Define a bounded, read-only HTTP client

Requests have timeouts, limited retries and a pagination limit. HTTP 400/404 fail immediately; selected transient errors retry. Pagination checks the source version and total row count. The simulated source is immutable. A real mutable API needs a snapshot/version contract or another extraction consistency strategy.

In [ ]:
def get_json(url):
    import json
    import time
    from urllib.request import Request, urlopen
    from urllib.error import HTTPError, URLError
    for attempt in range(3):
        try:
            with urlopen(Request(url, headers={"Accept":"application/json"}), timeout=5) as response:
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as error:
            if error.code not in {429,500,502,503,504} or attempt == 2:
                raise
            delay = min(float(error.headers.get("Retry-After", "1")), 2)
            error.close()
            time.sleep(max(0, delay))
        except (URLError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(0.5 * (2 ** attempt))

def fetch_pages(base_url, entity, version, country=None):
    from urllib.parse import urlencode
    page, seen, total, count = 1, set(), None, 0
    while page is not None:
        assert page not in seen and len(seen) < 100, "Invalid or unbounded pagination"
        seen.add(page)
        query = {"page":page, "page_size":2}
        if country is not None:
            query["country"] = country
        payload = get_json(base_url + "/v1/" + entity + "?" + urlencode(query))
        assert payload["dataset_version"] == version and payload["entity"] == entity
        assert payload["page"] == page and isinstance(payload["items"], list)
        if total is None:
            total = payload["total"]
        assert total == payload["total"], "Source changed during pagination"
        count += len(payload["items"])
        next_page = payload["next_page"]
        assert next_page is None or (type(next_page) is int and next_page == page + 1)
        yield payload
        page = next_page
    assert count == total, "Incomplete API extraction"

## Approach 1: pull pages on the driver

This is the easier approach for small or rate-limited APIs. Keep each complete response as a Bronze row. No seed values are recreated in this notebook: these records are fetched from notebook 01 over HTTP.

In [ ]:
version = get_json(API_BASE + "/health")["dataset_version"]
page_rows = []
counts = {}
for entity in ["products","customers","orders"]:
    pages = list(fetch_pages(API_BASE, entity, version))
    counts[entity] = sum(len(page["items"]) for page in pages)
    for page in pages:
        page_rows.append((entity, page["page"], version, json.dumps(page,sort_keys=True)))
assert counts == {"products":4,"customers":4,"orders":6}
assert get_json(API_BASE + "/health")["dataset_version"] == version
pages_df = spark.createDataFrame(page_rows, "entity STRING, page BIGINT, dataset_version STRING, response_json STRING")
assert pages_df.count() == 7
pages_df.select("entity","page","dataset_version").orderBy("entity","page").show()
pages_df.createOrReplaceTempView("incoming_api_pages")
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{LAB}.api_pages USING iceberg
    LOCATION '{LAKE_ROOT}/api_pages' TBLPROPERTIES ('format-version'='2')
    AS SELECT * FROM incoming_api_pages WHERE 1=0""")
pages_df.writeTo(f"polaris.{LAB}.api_pages").overwrite(F.lit(True))
assert spark.table(f"polaris.{LAB}.api_pages").count() == 7
print("PASS: seven HTTP response pages committed to Iceberg Bronze")

## Approach 2: execute HTTP GETs from Spark workers

The API supports disjoint country filters. Assign these filters to two Spark partitions, fetch each filter's pages and parse the returned record JSON with Spark. With the one-core workshop worker the tasks may run sequentially; the partition plan still limits concurrent requests to at most two.

Use an endpoint reachable from workers. Worker `localhost` refers to the worker, not the notebook API. Spark can retry tasks, so GET requests must tolerate repetition. Do not perform non-idempotent POST operations inside this pattern. Cache the small result to avoid unnecessary repeated HTTP calls during the example.

In [ ]:
def pull_country_partition(countries):
    import json
    for country in countries:
        for page in fetch_pages(API_BASE, "orders", version, country=country):
            for item in page["items"]:
                yield json.dumps(item)

raw_schema = "order_id BIGINT, customer_id BIGINT, product_id BIGINT, quantity INT, unit_price STRING, country STRING"
record_json = spark.sparkContext.parallelize(["IN","UK","US"], 2).mapPartitions(pull_country_partition)
orders_http = (spark.read.schema(raw_schema).json(record_json)
    .withColumn("unit_price", F.col("unit_price").cast("decimal(12,2)")).cache())
assert orders_http.count() == 6 and orders_http.select("order_id").distinct().count() == 6
assert orders_http.filter("unit_price IS NULL OR quantity <= 0").count() == 0
assert orders_http.select(F.sum(F.col("quantity")*F.col("unit_price"))).first()[0] == 190
orders_http.createOrReplaceTempView("incoming_http_orders")
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{LAB}.orders_parallel_http USING iceberg
    PARTITIONED BY (country) LOCATION '{LAKE_ROOT}/orders_parallel_http'
    TBLPROPERTIES ('format-version'='2') AS SELECT * FROM incoming_http_orders WHERE 1=0""")
orders_http.writeTo(f"polaris.{LAB}.orders_parallel_http").overwrite(F.lit(True))
orders_http.orderBy("order_id").show(truncate=False)
orders_http.unpersist()
print("PASS: worker HTTP extraction, six unique orders and value 190.00")

## Release Spark
Run this cell before switching Spark notebooks. Keep the separate API kernel alive until HTTP extraction finishes.

In [ ]:
spark.stop()